# Exoplanet Detection using 1D Convolutional Neural Networks
This notebook outlines the end-to-end pipeline for training a Deep Learning model to detect exoplanets using NASA's Kepler Space Telescope light curve data. 

The pipeline includes:
1. Data ingestion and preprocessing.
2. Handling severe class imbalance using SMOTE (Synthetic Minority Over-sampling Technique).
3. Data shuffling to ensure unbiased validation splits.
4. Building and training a 1D CNN tailored for sequential time-series data.

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

print("Initializing CNN pipeline for Exoplanet detection...")

# ==========================================
# 1. DATA LOADING
# ==========================================
print("Loading training data...")
# Ensure the path matches your repository structure
train_df = pd.read_csv('data/exoTrain.csv')

# Separate features and target variable
# LABEL: 2 represents an exoplanet star, 1 represents a non-exoplanet star.
# Subtracting 1 transforms labels to standard binary (1 = Exoplanet, 0 = Non-Exoplanet)
X_train = train_df.drop('LABEL', axis=1)
y_train = train_df['LABEL'] - 1

# ==========================================
# 2. PREPROCESSING & OVERSAMPLING
# ==========================================
print("Normalizing data and applying SMOTE...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Address class imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

# Reshape features to 3D for Conv1D input: (samples, time_steps, features)
X_train_cnn = np.expand_dims(X_train_bal, axis=2)

# ==========================================
# 3. DATA SHUFFLING
# ==========================================
print("Shuffling dataset to prevent validation bias...")
# Shuffling is strictly required after SMOTE to ensure a mixed validation split
indices = np.arange(X_train_cnn.shape[0])
np.random.shuffle(indices)
X_train_cnn = X_train_cnn[indices]
y_train_bal = y_train_bal[indices]

# ==========================================
# 4. MODEL ARCHITECTURE
# ==========================================
print("Building 1D Convolutional Neural Network...")
model = Sequential([
    # First Convolutional Block
    Conv1D(filters=32, kernel_size=5, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    
    # Second Convolutional Block
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    
    # Fully Connected Layer
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.2), # Dropout for regularization
    
    # Output Layer for binary classification
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# ==========================================
# 5. MODEL TRAINING & EXPORT
# ==========================================
print("Starting model training...")

# Batch size of 32 provides a good balance between training stability and memory footprint
history = model.fit(
    X_train_cnn, 
    y_train_bal, 
    epochs=5, 
    batch_size=32, 
    validation_split=0.2
)

# Save the compiled and trained model for inference in the Streamlit app
model_filename = "cnn_exoplanet_model.keras"
model.save(model_filename)
print(f"\n[SUCCESS] Training complete. Model exported as: {model_filename}")

C:\Users\romad\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Initializing CNN pipeline for Exoplanet detection...
Loading training data...
Normalizing data and applying SMOTE...
Shuffling dataset to prevent validation bias...
Building 1D Convolutional Neural Network...
Starting model training...
Epoch 1/5


C:\Users\romad\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


253/253 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.6546 - loss: 0.6112 - val_accuracy: 0.7975 - val_loss: 0.4919
Epoch 2/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8662 - loss: 0.3346 - val_accuracy: 0.9520 - val_loss: 0.2218
Epoch 3/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.9476 - loss: 0.1559 - val_accuracy: 0.9782 - val_loss: 0.1421
Epoch 4/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.9796 - loss: 0.0859 - val_accuracy: 0.9891 - val_loss: 0.0523
Epoch 5/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.9866 - loss: 0.0507 - val_accuracy: 0.9757 - val_loss: 0.1189

[SUCCESS] Training complete. Model exported as: cnn_exoplanet_model.keras
